In [1]:
import galois
import numpy as np

class BCH:
    def __init__(self, p, p_m, m, t):
        self.p = p
        self.p_m = p_m
        self.m = m
        self.t = t
        
        # 1. Define the Fields
        # Symbol Field: Where data lives (GF(9) if p=3, p_m=2)
        self.field_sym = galois.GF(p**p_m)
        # Extension Field: Where roots live (GF(9^m))
        self.field_ext = galois.GF((p**p_m)**m)
        
        self.n = self.field_ext.order - 1
        self.alpha = self.field_ext.primitive_element
        
        # 2. Setup Isomorphism Mapping (Hardware "Remapping")
        self.ext_to_sym, self.sym_to_ext = self._get_isomorphism()
        
        self.min_polys = []
        self.g = None
        self.k = 0
        
        self._initialize_generator()
        self.print_info()

    def _get_isomorphism(self):
        """Finds the mapping between field_ext subfield and field_sym."""
        q = self.field_sym.order
        # Find element in ext field that acts as primitive element for the subfield
        subfield_elements = self.field_ext.elements[np.where(self.field_ext.elements**q == self.field_ext.elements)]
        
        # Target: the irreducible poly that defined our symbol field
        target_min_poly = self.field_sym.irreducible_poly
        
        alpha_sub = None
        for element in subfield_elements:
            if element == 0: continue
            # Evaluate using extension field math
            if target_min_poly(element, field=self.field_ext) == 0:
                alpha_sub = element
                break
        
        if alpha_sub is None:
            raise ValueError("Isomorphism failed: No root found for subfield.")

        # Build Integer-based lookup tables (Dictionaries)
        ext_to_sym = {int(self.field_ext(0)): self.field_sym(0)}
        sym_to_ext = {int(self.field_sym(0)): self.field_ext(0)}
        
        curr_sub = self.field_ext(1)
        curr_sym = self.field_sym(1)
        for _ in range(q - 1):
            ext_to_sym[int(curr_sub)] = curr_sym
            sym_to_ext[int(curr_sym)] = curr_sub
            curr_sub = curr_sub * alpha_sub
            curr_sym = curr_sym * self.field_sym.primitive_element
            # this is bad  
            # curr_sub *= alpha_sub                               
            # curr_sym *= self.field_sym.primitive_element



        # print("ext_to_sym",ext_to_sym)
        # print("sym_to_ext",sym_to_ext)
        return ext_to_sym, sym_to_ext

    def _initialize_generator(self):
        """Calculates g(x) over the new Symbol Field."""
        current_g = galois.Poly([1], field=self.field_sym)
        seen_roots = set()
        q = self.field_sym.order # This is p^p_m
        
        for i in range(1, 2 * self.t + 1):
            root = self.alpha**i
            
            if int(root) not in seen_roots:
                # 1. Manually find the q-ary cyclotomic coset (the conjugates)
                conjugates = []
                curr = root
                while True:
                    conjugates.append(curr)
                    seen_roots.add(int(curr))
                    curr = curr**q
                    if curr == root: # We've looped back to the start
                        break
                
                # 2. Build the minimal polynomial: Product of (x - beta_j)
                # Math must be done in field_ext
                m_poly_ext = galois.Poly.Roots(conjugates, field=self.field_ext)
                
                # 3. Map coefficients from extension field back to symbol field
                # They are guaranteed to be in the subfield by Galois Theory
                for c in m_poly_ext.coeffs:
                    if int(c) not in self.ext_to_sym:
                        print(f"ERROR: {c} is not in field_sub")
                mapped_coeffs = [self.ext_to_sym[int(c)] for c in m_poly_ext.coeffs]
                m_poly = galois.Poly(mapped_coeffs, field=self.field_sym)
                
                current_g = current_g * m_poly

        self.g = current_g
        self.k = self.n - self.g.degree
    def print_info(self):
        print("-" * 40)
        print(f"BCH Code Configuration (GF({self.field_sym.order}^{self.m}))")
        print("-" * 40)
        print(f"n (Block Length):    {self.n}")
        print(f"k (Message Length):  {self.k}")
        print(f"t (Corrects errors): {self.t}")
        print(f"Parity Symbols:      {self.n - self.k}")
        print(f"Primitive Poly:      {self.field_ext.irreducible_poly}")
        print(f"\nUnique Minimal Polynomials used for g(x):")
        for idx, poly in enumerate(self.min_polys):
            print(f"  M_{idx+1}(x): {poly}")
        print(f"\nFinal Generator Polynomial g(x):")
        print(f"  {self.g}")
        print("-" * 40)
    def is_legal_codeword_division(self, received_word):
        r_poly = galois.Poly(received_word, field=self.field_sym)
        # self.g is your generator polynomial
        remainder = r_poly % self.g
        return remainder == 0
    def encode(self, message):
        """
        Takes a message of length k (list or array of integers 0..p-1)
        Returns a systematic codeword of length n.
        """
        if len(message) != self.k:
            raise ValueError(f"Message length must be {self.k}, but got {len(message)}")

        # Convert input to a galois.Poly over GF(p)
        msg_poly = galois.Poly(message, field=self.field_sym)
        
        # Shift message poly: m(x) * x^(n-k)
        parity_degree = self.n - self.k
        shifted_msg = msg_poly * galois.Poly.Degrees([parity_degree], field=self.field_sym)
        
        # Calculate parity: remainder of (shifted_msg / g)
        parity = shifted_msg % self.g
        
        # Systematic codeword = shifted_msg - parity
        codeword_poly = shifted_msg - parity
        
        # FIX: Access coefficients as a property and pad manually to length n
        # We pad with leading zeros because coefficients are returned from highest degree down
        coeffs = codeword_poly.coeffs
        padding_size = self.n - len(coeffs)
        codeword_coeffs = np.pad(coeffs, (padding_size, 0), 'constant', constant_values=0)
        
        return codeword_coeffs
    def inject_noise(self, codeword, num_errors=0, num_erasures=0):
        # 1. Ensure the base array is a FieldArray
        corrupted = self.field_sym(codeword).copy()
        
        all_indices = list(range(self.n))
        np.random.shuffle(all_indices)
        
        # Get all non-zero elements once to save time
        non_zero_elements = self.field_sym.elements[1:]
        
        # 2. Process Erasures
        erasure_indices = sorted(all_indices[:num_erasures])
        for idx in erasure_indices:
            # np.random.choice returns a scalar, we cast it back to field_sym
            noise = self.field_sym(np.random.choice(non_zero_elements))
            corrupted[idx] += noise
            
        # 3. Process Unknown Errors
        error_indices = all_indices[num_erasures : num_erasures + num_errors]
        for idx in error_indices:
            noise = self.field_sym(np.random.choice(non_zero_elements))
            corrupted[idx] += noise
            
        return corrupted, erasure_indices, error_indices
    def calculate_syndromes(self, received_word):
        """
        Calculates the 2t syndromes of the received word.
        S_i = R(alpha^i)
        """
        # Convert received symbols to a polynomial in the extension field
        # Note: coefficients are high-to-low degree
        received_ext = [self.sym_to_ext[int(s)] for s in received_word]
        r_poly = galois.Poly(received_ext, field=self.field_ext)
        
        syndromes = []
        # We need 2*t syndromes for the roots alpha^1 to alpha^2t
        for i in range(1, 2 * self.t + 1):
            root = self.alpha**i
            s_i = r_poly(root) # Evaluate polynomial at alpha^i
            syndromes.append(s_i)
            
        return self.field_ext(syndromes)

    def calculate_locator(self, erasure_indices):
        """
        Gamma(x) = Product of (1 - alpha^j * x) for all j in erasure_indices.
        Note: j is the index in the codeword (0 to n-1).
        Because our poly is high-to-low, index 'j' corresponds to x^(n-1-j).
        """
        gamma = galois.Poly([1], field=self.field_ext)
        for idx in erasure_indices:
            # The location in the polynomial sense is alpha^(n-1-idx)
            location = self.alpha**(self.n - 1 - idx)
            term = galois.Poly([location, -1], field=self.field_ext) # (location*x + 1)
            gamma = gamma * term
            
        return gamma
    def calculate_modified_syndromes(self, syndromes, erasure_locator):
        """
        S'(x) = S(x) * Gamma(x) mod x^(2t)
        The first deg(Gamma) syndromes are used to 'absorb' erasures.
        BMA only needs the remaining (2t - deg(Gamma)) syndromes.
        """
        # S(x) = S_1 + S_2x + ... + S_{2t}x^{2t-1}
        s_poly = galois.Poly(syndromes[::-1], field=self.field_ext)
        
        # Product S(x) * Gamma(x)
        combined_poly = s_poly * erasure_locator
        
        # In errors-and-erasures decoding, the 'useful' syndromes for 
        # finding unknown errors are S'_{rho+1} to S'_{2t}
        # where rho = deg(erasure_locator).
        rho = erasure_locator.degree
        target_len = 2 * self.t
        
        # Get all coefficients [x^(deg), ..., x^1, x^0]
        all_coeffs = combined_poly.coeffs
        
        # Pad to ensure we have at least target_len coefficients to pick from
        if len(all_coeffs) < target_len:
            all_coeffs = np.pad(all_coeffs, (target_len - len(all_coeffs), 0), 'constant')
            
        # Reverse to get [S'_1, S'_2, ..., S'_2t]
        s_modified_all = all_coeffs[::-1]
        
        # Slice from index rho to 2t
        # These are the syndromes S'_{rho+1} ... S'_{2t}
        final_syndromes = s_modified_all[rho:target_len]
        return self.field_ext(final_syndromes)
    def solve_bma(self, syndromes):
        """
        Berlekamp-Massey Algorithm for GF(p^m).
        Returns the Error Locator Polynomial Lambda(x).
        """
        t = len(syndromes)
        # Initial conditions
        lambda_poly = galois.Poly([1], field=self.field_ext)
        old_lambda = galois.Poly([1], field=self.field_ext)
        
        l = 0  # Current number of errors found
        m = 1  # Shift factor
        b = self.field_ext(1) # Previous discrepancy
        
        # Iterating through the 2t syndromes
        for n in range(t):
            # Calculate discrepancy (d)
            # d = S_{n+1} + sum_{i=1}^{L} lambda_i * S_{n+1-i}
            d = syndromes[n]
            lambda_coeffs = lambda_poly.coeffs[::-1] # low degree to high
            for i in range(1, len(lambda_coeffs)):
                if n - i >= 0:
                    d += lambda_coeffs[i] * syndromes[n - i]
            
            if d == 0:
                m += 1
            else:
                old_poly_shifted = old_lambda * galois.Poly.Degrees([m], field=self.field_ext)
                t_poly = lambda_poly - (d / b) * old_poly_shifted
                
                if 2 * l <= n:
                    old_lambda = lambda_poly
                    lambda_poly = t_poly
                    l = n + 1 - l
                    b = d
                    m = 1
                else:
                    lambda_poly = t_poly
                    m += 1
                    
        return lambda_poly
    def solve_eea(self, modified_syndromes, num_erasures):
        """
        EEA for Errors and Erasures.
        modified_syndromes: S'(x) = S(x) * Gamma(x) mod x^2t
        num_erasures: degree of Gamma(x)
        """
        # The number of syndromes we are working with is 2t
        # But we only have (2t - num_erasures) 'useful' syndromes left
        # deg_limit = 2 * self.t
        deg_limit = len(modified_syndromes)
        
        # r0 = x^(2t)
        r0 = galois.Poly.Degrees([deg_limit], field=self.field_ext)
        # r1 = S'(x). Note: modified_syndromes should be S'_1 to S'_2t
        r1 = galois.Poly(modified_syndromes[::-1], field=self.field_ext)
        
        v0 = galois.Poly([0], field=self.field_ext)
        v1 = galois.Poly([1], field=self.field_ext)
        
        # Stopping condition for errors + erasures:
        # We need to find unknown error locator Sigma(x)
        # Degree of Sigma(x) <= (2t - num_erasures) / 2
        target_degree = deg_limit // 2
        
        while r1.degree >= target_degree:
            q, r = divmod(r0, r1)
            v = v0 - q * v1
            
            r0, r1 = r1, r
            v0, v1 = v1, v
            
        # Lambda_total(x) = Sigma(x) * Gamma(x)
        # v1 here is the Sigma(x) (unknown error locator)
        # To get the final Lambda, we must multiply by erasure_locator outside
        # or pass erasure_locator in.
        
        # Normalize
        scaling_factor = v1.coeffs[-1]
        inv_scaling = self.field_ext(1) / scaling_factor
        
        sigma_poly = v1 * inv_scaling
        omega_poly = r1 * inv_scaling
        
        return sigma_poly, omega_poly
    def chien_search(self, lambda_poly):
        """
        Locates the errors by testing every possible position.
        The error locations are the indices i where Lambda(alpha^-i) = 0.
        """
        error_indices = []
        # We check every position from 0 to n-1
        # In polynomial terms, index i corresponds to alpha^(n-1-i)
        for i in range(self.n):
            # Evaluate at alpha**(-(n-1-i))
            # Mathematically equivalent to checking if alpha**(n-1-i) is a root
            inv_loc = self.alpha**(-(self.n - 1 - i))
            if lambda_poly(inv_loc) == 0:
                error_indices.append(i)
                
        return error_indices
    def derive_omega(self, lambda_poly, syndromes):
        """
        Derives Omega(x) = [Lambda(x) * S(x)] mod x^(2t)
        Works for both BMA and EEA outputs.
        """
        # 1. Convert syndrome list to a polynomial S(x) = S_1 + S_2x + ...
        s_poly = galois.Poly(syndromes[::-1], field=self.field_ext)
        
        # 2. Multiply Lambda(x) by S(x)
        combined = lambda_poly * s_poly
        
        # 3. Take modulo x^(2t)
        # We only keep terms from x^0 to x^(2t-1)
        mod_degree = 2 * self.t
        mod_poly = galois.Poly.Degrees([mod_degree], field=self.field_ext)
        _, omega_poly = divmod(combined, mod_poly)
        
        return omega_poly
    def forney_algorithm(self, omega_poly, lambda_poly, error_indices):
        """
        Calculates error magnitudes for given indices.
        error_indices: both unknown error positions and known erasures.
        """
        magnitudes = {}
        # Formal derivative of the total locator polynomial
        lambda_prime = lambda_poly.derivative()
        
        for idx in error_indices:
            # X_j^-1 is the value we plug into the polynomials
            # It is the root of the locator polynomial
            root = self.alpha**(-(self.n - 1 - idx))
            
            # Numerator: Omega(root)
            num = omega_poly(root)
            
            # Denominator: Lambda'(root)
            den = lambda_prime(root)
            
            # Magnitude Y_j = - (Omega / Lambda')
            # In Finite Fields, subtraction is addition of the additive inverse
            mag = -(num / den)
            magnitudes[idx] = mag
            
        return magnitudes
    # # --- UPDATED DECODE LOGIC ---
    # def decode(self, received_word, erasure_indices=[]):
    #     """Full Errors-and-Erasures Decoder for Extension Field Symbols."""
    #     # Ensure input is cast to the symbol field
    #     received_word = self.field_sym(received_word)
        
    #     # 1. Syndrome calculation (Convert symbols to extension field first)
    #     # We remap each symbol to its 'twin' in the extension field for the math
    #     received_ext = [self.sym_to_ext[int(s)] for s in received_word]
    #     syndromes = self.calculate_syndromes(received_ext)
        
    #     if np.all(syndromes == 0):
    #         return received_word, True

    #     # 2. Locator Polynomials
    #     gamma = self.calculate_locator(erasure_indices)
    #     s_prime = self.calculate_modified_syndromes(syndromes, gamma)
        
    #     # 3. Solve for unknown errors (BMA)
    #     sigma = self.solve_bma(s_prime)
    #     total_lambda = sigma * gamma
        
    #     # 4. Find Locations
    #     all_indices = self.chien_search(total_lambda)
        
    #     if len(all_indices) != total_lambda.degree:
    #         return received_word, False # Decoder Failure

    #     # 5. Forney Algorithm for Magnitudes
    #     omega = self.derive_omega(total_lambda, syndromes)
    #     magnitudes_ext = self.forney_algorithm(omega, total_lambda, all_indices)
        
    #     # 6. Final Correction with Isomorphism Downcasting
    #     corrected_word = np.copy(received_ext)
    #     for idx, mag_ext in magnitudes_ext.items():
    #         # Remap magnitude from extension field back to symbol field
    #         mag_sym = self.ext_to_sym[int(mag_ext)]
    #         corrected_word[idx] -= mag_sym
            
    #     return self.field_sym(corrected_word), True


In [2]:

# Create the codec
p = 2
p_m = 4
m =3
t = 12
bch_codec = BCH(p, p_m, m, t)
def product_view(bch, lambda_poly, syndromes, modified_syndromes, erasures_indices, error_indices):
    print("-" * 30)

    m_syndromes_poly = galois.Poly(modified_syndromes[::-1], field=bch.field_ext)
    syndromes_poly = galois.Poly(syndromes[::-1], field=bch.field_ext)


    error_locator = bch_codec.calculate_locator( error_indices)
    erasures_locator = bch_codec.calculate_locator( erasures_indices)
    # 2. Calculate lambda(x) * S'(x)
    lhs_gt = error_locator * m_syndromes_poly
    lhs = lambda_poly * m_syndromes_poly
    print(f"Ground truth : error_locator(x) * S'(x) coefficients: {lhs_gt.coeffs}")
    print(f"lambda(x) * S'(x) coefficients: {lhs.coeffs}")

    # 2. Calculate lambda(x) * S(x)
    lambda_gamma_syndromes_poly_gt = syndromes_poly * error_locator*erasures_locator
    lambda_gamma_syndromes_poly = syndromes_poly * lambda_poly*erasures_locator
    print(f"Ground truth : error_locator(x)*gamma(x)*S(x) coefficients: {lambda_gamma_syndromes_poly_gt.coeffs}")
    print(f"lambda(x)*gamma(x)*S(x) coefficients: {lambda_gamma_syndromes_poly.coeffs}")


    print("-" * 30)
for i in range(10):
    # 1. Encode
    msg = bch_codec.field_sym([np.random.randint(0, bch_codec.field_sym.order) for _ in range(bch_codec.k)])
    codeword = bch_codec.encode(msg)
    # 2. Inject Noise
    # Recall our rule: 2*errors + erasures < d_min
    num_errors = 4
    num_erasures = 3
    received, erasures, errors = bch_codec.inject_noise(codeword, num_errors=num_errors, num_erasures=num_erasures)
    # Verify that the received word is different from the codeword
    diff_count = np.sum(received != codeword)
    
    syndromes = bch_codec.calculate_syndromes(received_word=received)
    erasures_locator = bch_codec.calculate_locator(erasure_indices=erasures)
    modified_syndromes = bch_codec.calculate_modified_syndromes(syndromes=syndromes,erasure_locator=erasures_locator)
    lambda_bma = bch_codec.solve_bma(modified_syndromes)
    lambda_eea, omega_eea = bch_codec.solve_eea(modified_syndromes,num_erasures)
    error_pos = bch_codec.chien_search( lambda_bma)
    error_pos = sorted(error_pos)
    errors = sorted(errors)
    if lambda_bma != lambda_eea or error_pos!=errors:    

        print(f"Codeword: {codeword}") 
        print(f"Received: {received}")
        print(f"Erasure indices (known to decoder): {erasures}")
        print(f"Error indices (unknown to decoder): {errors}")
        print(f"Total corrupted symbols: {diff_count}")
        print(f"Syndromes is :{syndromes}")
        print(f"Erasures locator is :{erasures_locator}")
        print(f"Modified Syndromes is :{modified_syndromes}")
        print(f"BMA Lambda: {lambda_bma}")
        print(f"EEA Lambda: {lambda_eea}")
        print(f"Match? {lambda_bma == lambda_eea}")
        print(f"chien_search bma {bch_codec.chien_search( lambda_bma)}")
        print(f"chien_search eea {bch_codec.chien_search( lambda_eea)}")
        sigma_real = bch_codec.calculate_locator(errors)
        print(f"chien_search ground truth {bch_codec.chien_search( sigma_real)}")
        print(f"Ground truth error locator {sigma_real}")
        product_view(bch_codec, lambda_bma, syndromes, modified_syndromes, erasures, errors)
        break
    # product_view(bch_codec, lambda_bma, syndromes, modified_syndromes, erasures, errors)

    # 1. Total Locator: Combine Unknown Error Locator (sigma) and Erasure Locator (gamma)
    # sigma_eea comes from your EEA solver
    total_lambda = lambda_eea * erasures_locator

    # 2. Find ALL positions (Errors + Erasures)
    # In a perfect world, this should return exactly the indices you injected
    all_error_indices = error_pos+erasures
       

    omega=bch_codec.derive_omega(total_lambda, syndromes)

    # product_view(bch_codec, lambda_bma, syndromes, modified_syndromes, erasures, errors)
    
    # 3. Calculate Magnitudes
    # omega_eea also comes from your EEA solver
    error_values = bch_codec.forney_algorithm(omega, total_lambda, all_error_indices)

    # 4. Correct the Received Word
    # 1. Ensure the array itself is a Galois Field Array (GF(3))
    # If it's just a numpy array, the library will throw that TypeError
    corrected_word = bch_codec.field_sym(received) 
    codeword = bch_codec.field_sym(codeword)
    # 2. Iterate through and subtract
    for idx, val in error_values.items():
        # Downcast the extension field magnitude to the base field
        mag_sym = bch_codec.ext_to_sym[int(val)]
        # mag_base = bch_codec.field_sym(val)
        
        # Now both operands are GF(3) instances, and the library will be happy
        corrected_word[idx] -= mag_sym

    # 3. Check results
    # Note: codeword should also be a field_sym array for a direct comparison
    if np.array_equal(corrected_word, codeword):
        print(f"Correction Successful")
    else :
        
        print("Error")
        print("ext_to_sym",bch_codec.ext_to_sym)
        print("sym_to_ext",bch_codec.sym_to_ext)
        for idx, val in error_values.items():
            # Downcast the extension field magnitude to the base field
            mag_sym = bch_codec.ext_to_sym[int(val)]
            print(idx,mag_sym,val)
            
        print(f"chien_search eea {bch_codec.chien_search( lambda_eea)}")
        sigma_real = bch_codec.calculate_locator(errors)
        print(f"chien_search ground truth {bch_codec.chien_search( sigma_real)}")
        for ep in errors:
            print(ep,received[ep]-codeword[ep])
        break

----------------------------------------
BCH Code Configuration (GF(16^3))
----------------------------------------
n (Block Length):    4095
k (Message Length):  4026
t (Corrects errors): 12
Parity Symbols:      69
Primitive Poly:      x^12 + x^7 + x^6 + x^5 + x^3 + x + 1

Unique Minimal Polynomials used for g(x):

Final Generator Polynomial g(x):
  x^69 + 15x^68 + 13x^66 + 5x^65 + 5x^64 + 9x^63 + 3x^62 + 3x^61 + 6x^60 + 4x^59 + 12x^58 + 4x^57 + 14x^55 + 3x^54 + 14x^53 + 12x^52 + 14x^51 + 8x^50 + 7x^49 + 13x^48 + 3x^46 + 10x^45 + 14x^44 + 5x^43 + 5x^42 + 2x^41 + 2x^40 + x^39 + 9x^38 + 2x^36 + 2x^35 + 5x^34 + 2x^33 + 4x^32 + 4x^31 + 11x^30 + 9x^28 + 8x^27 + 13x^26 + 8x^25 + 8x^24 + 9x^23 + 5x^22 + 14x^21 + 8x^20 + 13x^19 + 5x^18 + 13x^17 + 12x^16 + 14x^15 + 3x^14 + 10x^13 + 9x^12 + 11x^11 + 12x^10 + 13x^9 + 2x^8 + 2x^7 + 5x^6 + 10x^5 + 11x^4 + 2x^2 + 9x + 9
----------------------------------------
Correction Successful
Correction Successful
Correction Successful
Correction Successful
C

In [3]:
def check_round_trip(bch):
    print("Checking Round Trip Mapping...")
    for i in range(bch.field_sym.order):
        sym_val = bch.field_sym(i)
        # Sym -> Ext -> Sym
        ext_val = bch.sym_to_ext[int(sym_val)]
        recovered_sym = bch.ext_to_sym[int(ext_val)]
        
        if sym_val != recovered_sym:
            print(f"[-] Round trip failed for {sym_val}!")
            return False
    print("[+] Round trip mapping is perfect.")
    return True
check_round_trip(bch=bch_codec)

Checking Round Trip Mapping...
[+] Round trip mapping is perfect.


True

In [4]:
import galois
import numpy as np

def get_improved_isomorphism(field_sym, field_ext):
    q = field_sym.order
    m_ext = field_ext.degree
    m_sym = field_sym.degree
    
    # 1. Get all elements in field_ext that belong to the subfield GF(q)
    # Elements satisfying x^q = x
    subfield_elements = field_ext.elements[np.where(field_ext.elements**q == field_ext.elements)]
    
    # 2. Get the minimal polynomial of the symbol field's primitive element
    # This polynomial defines the structure of GF(q)
    alpha_sym = field_sym.primitive_element
    min_poly = field_sym.conway_poly if hasattr(field_sym, "conway_poly") else field_sym.irreducible_poly
    
    # 3. Find an element in the subfield that is a root of this polynomial
    # This is the 'alpha_sub' that behaves EXACTLY like 'alpha_sym'
    alpha_sub = None
    for element in subfield_elements:
        if element == 0: continue
        # Evaluate min_poly(element) in the context of field_ext
        if min_poly(element, field=field_ext) == 0:
            alpha_sub = element
            break
            
    if alpha_sub is None:
        raise ValueError("Could not find a valid isomorphism root.")

    # 4. Construct the mapping table
    # Using a dictionary for O(1) translation during decoding
   # 4. Construct the mapping table using integers as keys
    ext_to_sym = {int(field_ext(0)): field_sym(0)}
    sym_to_ext = {int(field_sym(0)): field_ext(0)}
    
    current_sub = field_ext(1)
    current_sym = field_sym(1)
    
    for _ in range(q - 1):
        # Store using int() to avoid the unhashable TypeError
        ext_to_sym[int(current_sub)] = current_sym
        sym_to_ext[int(current_sym)] = current_sub
        
        current_sub *= alpha_sub
        current_sym *= alpha_sym
    
        
    return ext_to_sym, sym_to_ext

# Usage
p, p_m, m = 3, 2, 3
field_sym = galois.GF(p**p_m)
field_ext = galois.GF((p**p_m)**m)

ext_to_sym, sym_to_ext = get_improved_isomorphism(field_sym, field_ext)
print(f"Mapping found! Subfield generator in field_ext: {sym_to_ext}")

Mapping found! Subfield generator in field_ext: {0: GF(0, order=3^6), 1: GF(293, order=3^6), 3: GF(291, order=3^6), 4: GF(557, order=3^6), 7: GF(2, order=3^6), 2: GF(556, order=3^6), 6: GF(555, order=3^6), 8: GF(292, order=3^6), 5: GF(1, order=3^6)}
